# From a code on paper to a complete qodec

A quantum error-correcting code in a paper is a short list of Pauli operators. That is enough to reason about the code, but not enough to prepare, preserve, or read out an encoded state.

`qdk.ec.build_qodec` bridges the gap. Given a `qodec.Code`, it returns a complete, verified [qodec](https://github.com/microsoft/qodec): a logical instruction set over the code's logical qubits, a physical instruction set, and a synthesized gadget for every instruction.

This notebook takes the Steane code from its stabilizers to a complete qodec without writing a circuit by hand.

## Installing

```bash
pip install "qdk[ec]"
```

## 1. The code, as you would write it down

The Steane [[7,1,3]] code: seven physical qubits, one logical qubit, distance 3.
Six stabilizer generators — three X-type, three Z-type — and one logical X / Z
pair. This is the whole input.

In [ ]:
import qodec as qc

steane = qc.Code(
    "steane",
    stabilizers=[
        "X_0 X_3 X_4 X_6",
        "X_1 X_3 X_5 X_6",
        "X_2 X_4 X_5 X_6",
        "Z_0 Z_3 Z_4 Z_6",
        "Z_1 Z_3 Z_5 Z_6",
        "Z_2 Z_4 Z_5 Z_6",
    ],
    x=["X_0 X_1 X_3"],
    z=["Z_1 Z_2 Z_5"],
)

print(f"{len(list(steane.stabilizers))} stabilizers, {len(list(steane.x))} logical qubit(s)")


## 2. Synthesis

One call turns that into a runnable qodec.

In [ ]:
import qdk.ec as ec

protocol = ec.build_qodec(steane)
print(protocol.summary())

The result is a two-layer qodec. The top layer is a *synthesized* logical ISA —
instructions that talk about the logical qubit, not the seven physical ones —
and the bottom layer is the physical stim ISA the gadgets lower into.

In [ ]:
logical = protocol.layers[0]

for mnemonic, instruction in sorted(logical.isa.instructions.items()):
    print(f"{mnemonic:12s} {instruction.description}")

## 3. The circuits it wrote

`idle` is a syndrome-extraction round: one ancilla per stabilizer, each prepared
in |+>, coupled to its stabilizer's support with a controlled Pauli, then
rotated back and measured.

Note that `CX` is used where the stabilizer has an X, and `CZ` where it has a Z.
That one uniform construction handles CSS and non-CSS codes alike, and no data
qubit is ever touched by a basis-changing gate.

In [ ]:
print(logical.gadgets["idle"].circuit.source)

Readout is transversal, and the logical Pauli gadgets are just the code's own
logical operators applied gate by gate.

In [ ]:
for mnemonic in ("prepare_z", "measure_z", "measure_x", "x0", "z0"):
    source = logical.gadgets[mnemonic].circuit.source.strip().replace("\n", " ; ")
    print(f"{mnemonic:12s} {source[:78]}")

## 4. What makes it trustworthy

Synthesis does not assume that generated circuits are right. Each draft is completed through exact simulation, which discovers deterministic checks and logical readouts, then its realized channel is compared with the instruction's objective. With the default `strict=True`, any gadget that cannot be completed and verified raises instead of being silently omitted.

In [ ]:
idle = logical.gadgets["idle"]

print(f"{len(idle.checks)} checks discovered for `idle`; the first two:")
for check in list(idle.checks)[:2]:
    print("   ", [str(atom) for atom in check])

Second, every finished gadget is checked against the instruction it claims to
implement: the action its circuit *realizes* must equal the action the
instruction *declares*. Anything that fails is dropped rather than shipped, so a
gadget that survives is one whose circuit provably does what it says.

(Correctness is necessary but not sufficient — a circuit can implement the right
operation and still squander the code's protection. Section 5 measures that.)

In [ ]:
mismatches = {
    mnemonic: profile.action.why_not_equivalent_to(profile.objective)
    for mnemonic, gadget in logical.gadgets.items()
    if (profile := ec.GadgetProfile(gadget)).action.why_not_equivalent_to(profile.objective)
}
print("gadgets whose circuit disagrees with its objective:", mismatches or "none")

The code's distance survives the trip, and the full audit runs over the
synthesized qodec exactly as it would over a hand-authored one.

In [ ]:
code = ec.SubsystemCode.of(protocol.codes["steane"])
distance, witness = code.distance()
print("code distance:", distance, "| witness:", [str(p) for p in witness])

report = ec.audit(protocol)
print(f"audit: {len(report.errors)} error(s), {len(report.warnings)} warning(s)")
for diagnostic in report.errors:
    print("   ", diagnostic.rule, "|", diagnostic.summary)

> [!NOTE]
> An audit evaluates additional policy rules beyond synthesis's completion and channel-equivalence checks. Inspect every error before using the protocol.

## 5. Serializing it

The synthesized qodec is ordinary data — it serializes, round-trips, and is the
artifact you hand to a compilation pipeline. Nothing about it is second-class
compared to a hand-written one.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    path = Path(directory) / "steane.qodec.yaml"
    protocol.save(str(path), single_file=True)
    restored = qc.Qodec.load(str(path))

print("round-trips:", sorted(restored.layers[0].gadgets) == sorted(logical.gadgets))

## 6. When synthesis cannot finish the job

Not every construction works for every code. `build_qodec` defaults to `strict=True`, so it raises with the failing instruction instead of returning a protocol that silently omits part of its instruction set. Take the five-qubit code as it is conventionally written, with a logical Z that carries X components.

In [ ]:
FIVE_QUBIT_STABILIZERS = [
    "Z_0 X_1 X_2 Z_3",
    "Z_1 X_2 X_3 Z_4",
    "Z_0 Z_2 X_3 X_4",
    "X_0 Z_1 Z_3 X_4",
]

as_written = qc.Code(
    "five_qubit",
    stabilizers=FIVE_QUBIT_STABILIZERS,
    x=["X_0 X_1 X_2 X_3 X_4"],
    z=["X_0 X_3 Z_4"],
)

try:
    ec.build_qodec(as_written)
except ValueError as error:
    print("synthesis rejected the incomplete construction:")
    print(error)

## Where to go next

* Use `ec.build_qodec(code)` to synthesize a complete two-layer protocol.
* Use `ec.derive(artifact)` to complete a hand-authored gadget or qodec.
* Use `ec.GadgetProfile(gadget)` to inspect realized actions, checks, readouts, and fault effects.
* Use `ec.SubsystemCode.of(code)` for code algebra and distance calculations.
* Use `ec.audit(protocol)` for whole-protocol policy checks.

### Further reading

* Dennis, Kitaev, Landahl, and Preskill, *Topological quantum memory*, quant-ph/0110143, discusses hook errors.
* Chao and Reichardt, *Quantum error correction with only two extra qubits*, arXiv:1705.02329, describes the flag construction for distance-3 codes.
* Chamberland and Beverland, *Flag fault-tolerant error correction with arbitrary distance codes*, arXiv:1708.02246, generalizes the construction.

See `qdk_ec_walkthrough.ipynb` for the authoring, profiling, and testing lifecycle on a hand-authored qodec.